# 2DTFIM 2DNQS - (Nx,Ny)=(12,12): Inference (seed 222)

This is part of the work arxiv: 2606.25600 (Two-dimensional Hyperbolic RNN Neural Quantum State). For the purpose of reproducing the results, please check the link to the trained weight files. The saved weight links used in this notebook might not be the same as the ones in the Github repo. 

In [1]:
import sys
import os
sys.path.append('../../../utility_tfim')
from xdrnn_tfim2d_train_loop import *
import time
import glob

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=3.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped 

def define_load_test(wf, numsamples,path_to_weights, Ee, clipped_e = False):
    test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_before, wf)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=3.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy (not exact in 2D) is {np.round(Ee,4)}')

In [4]:
Nx=12
Ny=12
units =60
Jz=np.ones((Nx,Ny))
nsamples = 10000

seed=222
set_cpu_deterministic(seed)

# B=2.0

In [6]:
Bx=2.0
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'
E_dmrg=-346.98277872

In [7]:
import glob
# Search recursively for that specific filename
matches = glob.glob("../2DTFIM_2dNQS_res/(12, 12)_B=2.0/**/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=222_checkpoint.pt", recursive=True)
if matches:
    print(f"File found at: {matches}")
else:
    print("File is nowhere to be found in any subfolder.")

File found at: ['../2DTFIM_2dNQS_res/(12, 12)_B=2.0/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=222_checkpoint.pt']


### Euclidean 2DRNN

In [6]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.0422, var E = 262.8283
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.5865, var E = 2.328
DMRG energy (not exact in 2D) is -346.9828
Time taken =0.241 hrs


### Lorentz 2DRNN (L_max=2.0, 6.0)

In [15]:
# Search recursively for that specific filename
matches = glob.glob(f"../2DTFIM_2dNQS_res/(12, 12)_B=2.0/**/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed={seed}_checkpoint.pt", recursive=True)
if matches:
    print(f"File found at: {matches}")
else:
    print("File is nowhere to be found in any subfolder.")

File found at: ['../2DTFIM_2dNQS_res/(12, 12)_B=2.0/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed=222_checkpoint.pt']


In [11]:
sc=2.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.477, var E = 267.2661
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.2547, var E = 3.9155
DMRG energy (not exact in 2D) is -346.9828
Time taken =1.916 hrs


In [9]:
sc=6.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=6.0_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.1526, var E = 264.9692
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.4035, var E = 2.9605
DMRG energy (not exact in 2D) is -346.9828
Time taken =2.047 hrs


# B=3.0

In [5]:
Bx=3.0
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'
E_dmrg = -456.9066548624153

### Euclidean 2DRNN

In [9]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -431.8475, var E = 262.6895
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.8887, var E = 1.4945
DMRG energy (not exact in 2D) is -456.9067
Time taken =0.237 hrs


### Lorentz 2DRNN  (L_max=2.0, 5.0)

In [18]:
sc=2.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -432.0474, var E = 265.2717
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.8573, var E = 1.8049
DMRG energy (not exact in 2D) is -456.9067
Time taken =1.753 hrs


In [7]:
sc=5.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=5.0_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -432.0474, var E = 265.2717
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.7857, var E = 2.0881
DMRG energy (not exact in 2D) is -456.9067
Time taken =1.838 hrs


# B=4.0

In [10]:
Bx=4.0
E_dmrg=593.53890192
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'

### Euclidean 2DRNN

In [11]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.6528, var E = 263.3482
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -593.3814, var E = 2.5868
DMRG energy (not exact in 2D) is 593.5389
Time taken =0.172 hrs


### Lorentz 2DRNN (L_max=2.0, 1.5)

In [13]:
sc=1.5 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=1.5_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.8384, var E = 264.5204
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -593.4719, var E = 0.4611
DMRG energy (not exact in 2D) is 593.5389
Time taken =1.882 hrs


In [11]:
sc=2.0
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed={seed}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.8719, var E = 264.0716
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -593.4909, var E = 0.5994
DMRG energy (not exact in 2D) is 593.5389
Time taken =2.102 hrs
